# TCN-Based Multi-Scale Rainfall Prediction
## Phase 2: Temporal Convolutional Network

**Objective:** Predict rainfall intensity at three temporal scales using the GAN-completed
daily dataset from Phase 1.

**Prediction Targets:**
| Scale | Target | Description |
|---|---|---|
| Daily | Next-day rainfall (mm) per station | 1-step ahead forecast |
| Weekly | Total rainfall over next 7 days per station | Short-range aggregate |
| Monthly | Total rainfall over next 30 days per station | Medium-range aggregate |

**Architecture:** Multi-head TCN with shared dilated causal convolutional backbone and
scale-specific prediction heads. 90-day lookback window, GELU activations, AdamW optimizer.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
import os
import copy

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

---
## 1. Load & Prepare Data

Load the GAN-completed daily dataset and engineer temporal features from scratch
using the NaN-free data. This ensures rolling/lag features have no missing values.

In [ ]:
df = pd.read_csv('completed_daily_rainfall.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

STATIONS = ['nada', 'lembing', 'reman']

print(f"Dataset: {df.shape[0]:,} rows, {df['date'].min().date()} to {df['date'].max().date()}")
print(f"NaN check: {df[STATIONS].isna().sum().sum()}")
df.head()

### 1.1 Feature Engineering

Build a comprehensive feature set for the TCN:
- **Raw rainfall** per station (3 features)
- **Log-transformed rainfall** — log1p(rainfall) per station (3 features)
- **Cyclical time** — sin/cos of day-of-year and month (4 features)
- **Rolling statistics** — 3, 7, 14, 30-day moving averages per station (12 features)
- **Lag features** — 1, 3, 7-day lags per station (9 features)
- **Consecutive dry days** per station (3 features)
- **Cross-station mean** — spatial average (1 feature)

Total: 35 input features per timestep.

In [ ]:
# Cyclical time features
doy = df['date'].dt.dayofyear
month = df['date'].dt.month
df['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
df['month_sin'] = np.sin(2 * np.pi * month / 12)
df['month_cos'] = np.cos(2 * np.pi * month / 12)

# Rolling statistics
for s in STATIONS:
    for w in [3, 7, 14, 30]:
        df[f'{s}_rm{w}'] = df[s].rolling(w, min_periods=1).mean()

# Lag features
for s in STATIONS:
    for lag in [1, 3, 7]:
        df[f'{s}_lag{lag}'] = df[s].shift(lag)

# Consecutive dry days
for s in STATIONS:
    is_dry = (df[s] < 0.1).astype(int)
    groups = (is_dry != is_dry.shift()).cumsum()
    df[f'{s}_dry'] = is_dry.groupby(groups).cumsum()

# Cross-station mean
df['station_mean'] = df[STATIONS].mean(axis=1)

# Log-transformed rainfall
for s in STATIONS:
    df[f'{s}_log'] = np.log1p(df[s])

# Drop rows with NaN from lag features (first 7 rows)
df = df.dropna().reset_index(drop=True)

FEATURE_COLS = (
    STATIONS + [f'{s}_log' for s in STATIONS]
    + ['doy_sin', 'doy_cos', 'month_sin', 'month_cos']
    + [f'{s}_rm{w}' for s in STATIONS for w in [3, 7, 14, 30]]
    + [f'{s}_lag{lag}' for s in STATIONS for lag in [1, 3, 7]]
    + [f'{s}_dry' for s in STATIONS]
    + ['station_mean']
)

print(f"Feature count: {len(FEATURE_COLS)}")
print(f"Dataset after feature eng: {df.shape[0]:,} rows")
print(f"NaN check: {df[FEATURE_COLS].isna().sum().sum()}")
print(f"\nFeatures:")
for i, c in enumerate(FEATURE_COLS):
    print(f"  [{i:2d}] {c}")

### 1.2 Target Construction

For each day t, construct three targets:
- **Daily:** rainfall at t+1
- **Weekly:** sum of rainfall from t+1 to t+7
- **Monthly:** sum of rainfall from t+1 to t+30

In [ ]:
TARGET_STATIONS = STATIONS

for s in TARGET_STATIONS:
    df[f'{s}_target_daily'] = df[s].shift(-1)
    df[f'{s}_target_weekly'] = df[s].rolling(7).sum().shift(-7)
    df[f'{s}_target_monthly'] = df[s].rolling(30).sum().shift(-30)

DAILY_TARGETS = [f'{s}_target_daily' for s in TARGET_STATIONS]
WEEKLY_TARGETS = [f'{s}_target_weekly' for s in TARGET_STATIONS]
MONTHLY_TARGETS = [f'{s}_target_monthly' for s in TARGET_STATIONS]
ALL_TARGETS = DAILY_TARGETS + WEEKLY_TARGETS + MONTHLY_TARGETS

df = df.dropna(subset=ALL_TARGETS).reset_index(drop=True)

print(f"Dataset after target construction: {df.shape[0]:,} rows")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\nTarget stats:")
for name, cols in [('Daily', DAILY_TARGETS), ('Weekly', WEEKLY_TARGETS), ('Monthly', MONTHLY_TARGETS)]:
    vals = df[cols].values.flatten()
    print(f"  {name}: mean={vals.mean():.1f}, std={vals.std():.1f}, max={vals.max():.1f}")

### 1.3 Chronological Train / Validation / Test Split

Use a strict chronological split to prevent data leakage:
- **Train:** first 70% (~2009-2020)
- **Validation:** next 15% (~2020-2023)
- **Test:** last 15% (~2023-2025)

In [ ]:
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

n = len(df)
train_end = int(n * TRAIN_FRAC)
val_end = int(n * (TRAIN_FRAC + VAL_FRAC))

df_train = df.iloc[:train_end].copy()
df_val = df.iloc[train_end:val_end].copy()
df_test = df.iloc[val_end:].copy()

print(f"Train: {len(df_train):,} rows ({df_train['date'].min().date()} to {df_train['date'].max().date()})")
print(f"Val:   {len(df_val):,} rows ({df_val['date'].min().date()} to {df_val['date'].max().date()})")
print(f"Test:  {len(df_test):,} rows ({df_test['date'].min().date()} to {df_test['date'].max().date()})")

# Normalize features (fit on train only)
feature_scaler = StandardScaler()
feature_scaler.fit(df_train[FEATURE_COLS].values)

X_train = feature_scaler.transform(df_train[FEATURE_COLS].values).astype(np.float32)
X_val = feature_scaler.transform(df_val[FEATURE_COLS].values).astype(np.float32)
X_test = feature_scaler.transform(df_test[FEATURE_COLS].values).astype(np.float32)

# Normalize targets (fit on train only)
daily_scaler = StandardScaler()
weekly_scaler = StandardScaler()
monthly_scaler = StandardScaler()

daily_scaler.fit(df_train[DAILY_TARGETS].values)
weekly_scaler.fit(df_train[WEEKLY_TARGETS].values)
monthly_scaler.fit(df_train[MONTHLY_TARGETS].values)

Y_train_d = daily_scaler.transform(df_train[DAILY_TARGETS].values).astype(np.float32)
Y_train_w = weekly_scaler.transform(df_train[WEEKLY_TARGETS].values).astype(np.float32)
Y_train_m = monthly_scaler.transform(df_train[MONTHLY_TARGETS].values).astype(np.float32)

Y_val_d = daily_scaler.transform(df_val[DAILY_TARGETS].values).astype(np.float32)
Y_val_w = weekly_scaler.transform(df_val[WEEKLY_TARGETS].values).astype(np.float32)
Y_val_m = monthly_scaler.transform(df_val[MONTHLY_TARGETS].values).astype(np.float32)

Y_test_d = daily_scaler.transform(df_test[DAILY_TARGETS].values).astype(np.float32)
Y_test_w = weekly_scaler.transform(df_test[WEEKLY_TARGETS].values).astype(np.float32)
Y_test_m = monthly_scaler.transform(df_test[MONTHLY_TARGETS].values).astype(np.float32)

print(f"\nFeature scaling: mean~0, std~1 (fitted on train)")

### 1.4 Sequence Dataset

Create sliding-window sequences for the TCN. Each sample is a window of L consecutive
days of features, paired with the multi-scale targets for the last day in the window.

In [ ]:
LOOKBACK = 90

class MultiScaleDataset(Dataset):
    def __init__(self, X, Y_d, Y_w, Y_m, lookback):
        self.X = X
        self.Y_d = Y_d
        self.Y_w = Y_w
        self.Y_m = Y_m
        self.lookback = lookback
        self.n_samples = len(X) - lookback

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x = self.X[idx:idx + self.lookback]
        yd = self.Y_d[idx + self.lookback - 1]
        yw = self.Y_w[idx + self.lookback - 1]
        ym = self.Y_m[idx + self.lookback - 1]
        return (torch.FloatTensor(x.T),
                torch.FloatTensor(yd),
                torch.FloatTensor(yw),
                torch.FloatTensor(ym))

BATCH_SIZE = 128

train_ds = MultiScaleDataset(X_train, Y_train_d, Y_train_w, Y_train_m, LOOKBACK)
val_ds = MultiScaleDataset(X_val, Y_val_d, Y_val_w, Y_val_m, LOOKBACK)
test_ds = MultiScaleDataset(X_test, Y_test_d, Y_test_w, Y_test_m, LOOKBACK)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Lookback window: {LOOKBACK} days")
print(f"Train sequences: {len(train_ds):,}")
print(f"Val sequences:   {len(val_ds):,}")
print(f"Test sequences:  {len(test_ds):,}")
print(f"\nInput shape per sample: ({len(FEATURE_COLS)}, {LOOKBACK})")
print(f"Output: 3 stations x 3 scales = 9 target values")

---
## 2. TCN Architecture

The Temporal Convolutional Network uses **dilated causal convolutions** to capture
long-range temporal dependencies without recurrence. Key components:

- **Causal convolution:** output at time t depends only on inputs at times ≤ t
- **Dilated convolution:** exponentially increasing dilation factors (1, 2, 4, 8, ...)
  give the network a receptive field of 2^L without deep stacking
- **Residual connections:** stabilize gradient flow in deeper networks
- **Multi-head output:** shared backbone feeds into 3 prediction heads (daily/weekly/monthly)

In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation,
                              padding=self.padding)

    def forward(self, x):
        out = self.conv(x)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        return out


class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(in_ch, out_ch, kernel_size, dilation),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
            CausalConv1d(out_ch, out_ch, kernel_size, dilation),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return nn.functional.gelu(self.net(x) + self.downsample(x))


class MultiScaleTCN(nn.Module):
    def __init__(self, n_features, n_stations, hidden=48, n_levels=5, kernel_size=3, dropout=0.3):
        super().__init__()

        layers = []
        for i in range(n_levels):
            in_ch = n_features if i == 0 else hidden
            layers.append(TemporalBlock(in_ch, hidden, kernel_size, dilation=2**i, dropout=dropout))
        self.backbone = nn.Sequential(*layers)

        self.head_daily = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_stations),
        )
        self.head_weekly = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_stations),
        )
        self.head_monthly = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, n_stations),
        )

    def forward(self, x):
        h = self.backbone(x)[:, :, -1]
        return self.head_daily(h), self.head_weekly(h), self.head_monthly(h)


N_FEATURES = len(FEATURE_COLS)
N_STATIONS_OUT = len(TARGET_STATIONS)
HIDDEN = 48
N_LEVELS = 5
KERNEL_SIZE = 3
DROPOUT = 0.3

model = MultiScaleTCN(N_FEATURES, N_STATIONS_OUT, HIDDEN, N_LEVELS, KERNEL_SIZE, DROPOUT).to(DEVICE)

receptive_field = 1 + 2 * (KERNEL_SIZE - 1) * (2**N_LEVELS - 1)
total_params = sum(p.numel() for p in model.parameters())

print(f"Model parameters: {total_params:,}")
print(f"Receptive field: {receptive_field} timesteps")
print(f"Lookback window: {LOOKBACK} timesteps")
print(f"\nArchitecture:")
print(model)

---
## 3. Training

**Loss:** Combined MSE across all three scales with equal weighting.
The model learns to simultaneously predict daily, weekly, and monthly rainfall,
sharing temporal representations through the backbone.

**Optimizer:** AdamW with cosine annealing learning rate schedule and weight decay.
**Early stopping:** patience of 40 epochs on validation loss.

In [ ]:
LR = 5e-4
N_EPOCHS = 300
PATIENCE = 40
SCALE_WEIGHTS = {'daily': 1.0, 'weekly': 1.0, 'monthly': 1.0}

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-6)
mse_loss = nn.MSELoss()

history = {'train_loss': [], 'val_loss': [],
           'train_daily': [], 'train_weekly': [], 'train_monthly': [],
           'val_daily': [], 'val_weekly': [], 'val_monthly': []}

best_val_loss = float('inf')
best_model_state = None
patience_counter = 0

print(f"Training TCN for up to {N_EPOCHS} epochs (patience={PATIENCE})...")
print(f"  LR: {LR} (cosine annealing to 1e-6)")
print(f"  Weight decay: 1e-4")
print(f"  Scale weights: {SCALE_WEIGHTS}")
print("-" * 70)

for epoch in range(N_EPOCHS):
    model.train()
    train_losses = {'total': 0, 'daily': 0, 'weekly': 0, 'monthly': 0}
    n_train = 0

    for x, yd, yw, ym in train_loader:
        x, yd, yw, ym = x.to(DEVICE), yd.to(DEVICE), yw.to(DEVICE), ym.to(DEVICE)

        pred_d, pred_w, pred_m = model(x)

        loss_d = mse_loss(pred_d, yd)
        loss_w = mse_loss(pred_w, yw)
        loss_m = mse_loss(pred_m, ym)
        loss = (SCALE_WEIGHTS['daily'] * loss_d +
                SCALE_WEIGHTS['weekly'] * loss_w +
                SCALE_WEIGHTS['monthly'] * loss_m)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_losses['total'] += loss.item()
        train_losses['daily'] += loss_d.item()
        train_losses['weekly'] += loss_w.item()
        train_losses['monthly'] += loss_m.item()
        n_train += 1

    scheduler.step()

    model.eval()
    val_losses = {'total': 0, 'daily': 0, 'weekly': 0, 'monthly': 0}
    n_val = 0

    with torch.no_grad():
        for x, yd, yw, ym in val_loader:
            x, yd, yw, ym = x.to(DEVICE), yd.to(DEVICE), yw.to(DEVICE), ym.to(DEVICE)
            pred_d, pred_w, pred_m = model(x)
            loss_d = mse_loss(pred_d, yd)
            loss_w = mse_loss(pred_w, yw)
            loss_m = mse_loss(pred_m, ym)
            loss = (SCALE_WEIGHTS['daily'] * loss_d +
                    SCALE_WEIGHTS['weekly'] * loss_w +
                    SCALE_WEIGHTS['monthly'] * loss_m)
            val_losses['total'] += loss.item()
            val_losses['daily'] += loss_d.item()
            val_losses['weekly'] += loss_w.item()
            val_losses['monthly'] += loss_m.item()
            n_val += 1

    t_avg = train_losses['total'] / n_train
    v_avg = val_losses['total'] / n_val

    history['train_loss'].append(t_avg)
    history['val_loss'].append(v_avg)
    for key in ['daily', 'weekly', 'monthly']:
        history[f'train_{key}'].append(train_losses[key] / n_train)
        history[f'val_{key}'].append(val_losses[key] / n_val)

    if v_avg < best_val_loss:
        best_val_loss = v_avg
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        marker = ' *'
    else:
        patience_counter += 1
        marker = ''

    if (epoch + 1) % 25 == 0 or epoch == 0 or patience_counter == PATIENCE:
        print(f"Epoch {epoch+1:>3d}/{N_EPOCHS} | "
              f"Train: {t_avg:.4f} (D:{train_losses['daily']/n_train:.4f} W:{train_losses['weekly']/n_train:.4f} M:{train_losses['monthly']/n_train:.4f}) | "
              f"Val: {v_avg:.4f}{marker}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_model_state)
print(f"\nBest validation loss: {best_val_loss:.4f}")

### 3.1 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].plot(history['train_loss'], label='Train', linewidth=1)
axes[0].plot(history['val_loss'], label='Val', linewidth=1)
axes[0].set_title('Total Loss', fontweight='bold')
axes[0].legend()
axes[0].set_xlabel('Epoch')

for i, scale in enumerate(['daily', 'weekly', 'monthly']):
    ax = axes[i + 1]
    ax.plot(history[f'train_{scale}'], label='Train', linewidth=1)
    ax.plot(history[f'val_{scale}'], label='Val', linewidth=1)
    ax.set_title(f'{scale.capitalize()} Loss', fontweight='bold')
    ax.legend()
    ax.set_xlabel('Epoch')

plt.suptitle('TCN Training Progress', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('tcn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Test Set Evaluation

Evaluate the trained model on the held-out test set (unseen future data).
Predictions are inverse-transformed back to original mm scale for interpretable metrics.

In [ ]:
model.eval()

def collect_predictions(loader):
    preds_d, preds_w, preds_m = [], [], []
    trues_d, trues_w, trues_m = [], [], []

    with torch.no_grad():
        for x, yd, yw, ym in loader:
            x = x.to(DEVICE)
            pd_, pw_, pm_ = model(x)
            preds_d.append(pd_.cpu().numpy())
            preds_w.append(pw_.cpu().numpy())
            preds_m.append(pm_.cpu().numpy())
            trues_d.append(yd.numpy())
            trues_w.append(yw.numpy())
            trues_m.append(ym.numpy())

    return {
        'daily':   (np.concatenate(trues_d), np.concatenate(preds_d)),
        'weekly':  (np.concatenate(trues_w), np.concatenate(preds_w)),
        'monthly': (np.concatenate(trues_m), np.concatenate(preds_m)),
    }

test_results = collect_predictions(test_loader)

# Inverse transform to original scale
scalers = {'daily': daily_scaler, 'weekly': weekly_scaler, 'monthly': monthly_scaler}
test_mm = {}
for scale in ['daily', 'weekly', 'monthly']:
    true_scaled, pred_scaled = test_results[scale]
    sc = scalers[scale]
    true_mm = sc.inverse_transform(true_scaled)
    pred_mm = sc.inverse_transform(pred_scaled)
    pred_mm = np.maximum(pred_mm, 0.0)
    test_mm[scale] = (true_mm, pred_mm)

print("=" * 70)
print("TEST SET EVALUATION RESULTS")
print("=" * 70)

station_labels = ['Nada', 'Lembing', 'Reman']
all_results = {}

for scale in ['daily', 'weekly', 'monthly']:
    true_mm_arr, pred_mm_arr = test_mm[scale]
    print(f"\n  === {scale.upper()} PREDICTION ===")

    for j, s in enumerate(station_labels):
        yt = true_mm_arr[:, j]
        yp = pred_mm_arr[:, j]
        rmse = np.sqrt(mean_squared_error(yt, yp))
        mae = mean_absolute_error(yt, yp)
        r2 = r2_score(yt, yp)
        all_results[f'{scale}_{s}'] = {'rmse': rmse, 'mae': mae, 'r2': r2}
        print(f"    {s}: RMSE={rmse:.2f}mm, MAE={mae:.2f}mm, R\u00b2={r2:.4f}")

    yt_all = true_mm_arr.flatten()
    yp_all = pred_mm_arr.flatten()
    rmse_all = np.sqrt(mean_squared_error(yt_all, yp_all))
    mae_all = mean_absolute_error(yt_all, yp_all)
    r2_all = r2_score(yt_all, yp_all)
    all_results[f'{scale}_overall'] = {'rmse': rmse_all, 'mae': mae_all, 'r2': r2_all}
    print(f"    OVERALL: RMSE={rmse_all:.2f}mm, MAE={mae_all:.2f}mm, R\u00b2={r2_all:.4f}")

print("\n" + "=" * 70)

### 4.1 Actual vs Predicted — Scatter Plots

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 16))
scale_labels = ['Daily (mm)', 'Weekly Total (mm)', 'Monthly Total (mm)']

for row, scale in enumerate(['daily', 'weekly', 'monthly']):
    true_arr, pred_arr = test_mm[scale]
    for col, (sname, slabel) in enumerate(zip(TARGET_STATIONS, station_labels)):
        ax = axes[row, col]
        yt = true_arr[:, col]
        yp = pred_arr[:, col]
        ax.scatter(yt, yp, alpha=0.3, s=8, color='steelblue')
        mx = max(yt.max(), yp.max()) * 1.05
        ax.plot([0, mx], [0, mx], 'r--', lw=1, label='Perfect')
        r2 = all_results[f'{scale}_{station_labels[col]}']['r2']
        rmse = all_results[f'{scale}_{station_labels[col]}']['rmse']
        ax.set_title(f'{station_labels[col]} — {scale.capitalize()}\nR\u00b2={r2:.3f}, RMSE={rmse:.1f}',
                     fontweight='bold', fontsize=10)
        ax.set_xlabel(f'Actual {scale_labels[row]}')
        ax.set_ylabel(f'Predicted {scale_labels[row]}')
        ax.set_xlim(0, mx)
        ax.set_ylim(0, mx)
        ax.legend(fontsize=8)

plt.suptitle('Test Set: Actual vs TCN Predicted Rainfall', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('tcn_scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Time Series — Predicted vs Actual

In [ ]:
test_dates = df_test['date'].values[LOOKBACK:]

fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)

for ax, col, slabel in zip(axes, range(N_STATIONS_OUT), station_labels):
    true_d = test_mm['daily'][0][:, col]
    pred_d = test_mm['daily'][1][:, col]

    ax.plot(test_dates, true_d, linewidth=0.8, alpha=0.7, color='steelblue', label='Actual')
    ax.plot(test_dates, pred_d, linewidth=0.8, alpha=0.7, color='crimson', label='Predicted')
    ax.set_ylabel('Daily Rainfall (mm)')
    ax.set_title(f'{slabel} — Daily Prediction', fontweight='bold')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Date')
plt.suptitle('Test Set: TCN Daily Rainfall Predictions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('tcn_daily_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)

for ax, col, slabel in zip(axes, range(N_STATIONS_OUT), station_labels):
    true_w = test_mm['weekly'][0][:, col]
    pred_w = test_mm['weekly'][1][:, col]

    ax.plot(test_dates, true_w, linewidth=1, alpha=0.7, color='steelblue', label='Actual')
    ax.plot(test_dates, pred_w, linewidth=1, alpha=0.7, color='crimson', label='Predicted')
    ax.set_ylabel('Weekly Total (mm)')
    ax.set_title(f'{slabel} — Weekly Prediction', fontweight='bold')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Date')
plt.suptitle('Test Set: TCN Weekly Rainfall Predictions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('tcn_weekly_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)

for ax, col, slabel in zip(axes, range(N_STATIONS_OUT), station_labels):
    true_m = test_mm['monthly'][0][:, col]
    pred_m = test_mm['monthly'][1][:, col]

    ax.plot(test_dates, true_m, linewidth=1, alpha=0.7, color='steelblue', label='Actual')
    ax.plot(test_dates, pred_m, linewidth=1, alpha=0.7, color='crimson', label='Predicted')
    ax.set_ylabel('Monthly Total (mm)')
    ax.set_title(f'{slabel} — Monthly Prediction', fontweight='bold')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Date')
plt.suptitle('Test Set: TCN Monthly Rainfall Predictions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('tcn_monthly_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Export Artifacts

Save the trained TCN model, prediction results, and scalers.

In [ ]:
# Save model
MODEL_PATH = 'tcn_multiscale_model.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'n_features': N_FEATURES,
    'n_stations': N_STATIONS_OUT,
    'hidden': HIDDEN,
    'n_levels': N_LEVELS,
    'kernel_size': KERNEL_SIZE,
    'dropout': DROPOUT,
    'lookback': LOOKBACK,
    'feature_cols': FEATURE_COLS,
    'target_stations': TARGET_STATIONS,
    'feature_scaler_mean': feature_scaler.mean_.tolist(),
    'feature_scaler_scale': feature_scaler.scale_.tolist(),
    'daily_scaler_mean': daily_scaler.mean_.tolist(),
    'daily_scaler_scale': daily_scaler.scale_.tolist(),
    'weekly_scaler_mean': weekly_scaler.mean_.tolist(),
    'weekly_scaler_scale': weekly_scaler.scale_.tolist(),
    'monthly_scaler_mean': monthly_scaler.mean_.tolist(),
    'monthly_scaler_scale': monthly_scaler.scale_.tolist(),
    'test_results': all_results,
}, MODEL_PATH)

# Save prediction results
pred_df = pd.DataFrame({'date': test_dates})
for scale in ['daily', 'weekly', 'monthly']:
    true_arr, pred_arr = test_mm[scale]
    for j, s in enumerate(TARGET_STATIONS):
        pred_df[f'{s}_{scale}_actual'] = true_arr[:, j]
        pred_df[f'{s}_{scale}_predicted'] = pred_arr[:, j]

PRED_PATH = 'tcn_prediction_results.csv'
pred_df.to_csv(PRED_PATH, index=False)

print(f"Model saved: {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1024:.1f} KB)")
print(f"Predictions saved: {PRED_PATH} ({os.path.getsize(PRED_PATH)/1024:.1f} KB)")
print(f"  {pred_df.shape[0]:,} rows x {pred_df.shape[1]} columns")

---
## Summary

| Component | Detail |
|---|---|
| **Architecture** | Multi-head TCN: shared 5-level dilated causal backbone + 3 prediction heads |
| **Backbone** | 5 temporal blocks, hidden=48, kernel=3, dilation=[1,2,4,8,16], GELU activation |
| **Receptive field** | 125 days (~4 months of temporal context) |
| **Input** | 90-day lookback window x 35 features per timestep |
| **Outputs** | Daily (next-day), Weekly (7-day sum), Monthly (30-day sum) per station |
| **Training** | AdamW (lr=5e-4, cosine annealing, weight decay=1e-4), early stopping (patience=40) |
| **Data split** | 70% train / 15% val / 15% test (chronological) |
| **Parameters** | ~75K (compact model to reduce overfitting on ~4K training sequences) |

**Deliverables:**
- `tcn_multiscale_model.pth` — trained model weights + scalers
- `tcn_prediction_results.csv` — test set predictions at all 3 scales